## 《Flow-DPPO: Divergence Proximal Policy Optimization for Flow Matching Models》

https://arxiv.org/pdf/2606.11025

---

## 1 Introduction

**背景：** 在线强化学习（RL）已被证明能大幅提升流匹配模型在图像和视频生成中的质量和对齐度。Flow-GRPO 等方法将去噪过程建模为马尔可夫决策过程（MDP），并使用 PPO 式的比率裁剪来强制信任域约束。

**问题陈述：** 论文的核心论点直接而尖锐——**比率裁剪在流模型中存在结构性不适配**。具体表现为：

- 新旧策略之间的概率比率 $r^i_t(\theta)$，本质上是**真实策略散度的一个单样本、有噪声的蒙特卡洛估计**。
- 在高维连续潜在空间中，这个估计噪声被极度放大，导致：
  - 在轨迹的某些区域**过度约束**（不该裁剪的被裁剪了）
  - 在另一些区域**约束不足**（该裁剪的没裁剪住）
- 这进一步导致比率分布出现系统性左偏（均值低于1），使标准的 $[1-\epsilon, 1+\epsilon]$ 裁剪区间变得实际不对称，对正优势样本约束过松，对负优势样本约束过紧。

**核心观察：** 流模型有一个被忽视的结构性优势——每步策略是高斯分布，均值由速度网络决定，方差是固定的、依赖于时间调的。因此，**新旧策略之间的 KL 散度可以精确、零额外成本地计算**：
$$
D_{KL}(\pi_{\theta_{old}} \| \pi_\theta) = \frac{\|\mu_{\theta_{old}} - \mu_\theta\|^2}{2\sigma^2}
$$
这完全不同于大语言模型的场景（需要在巨大词表上做近似），流模型的散度计算是免费的、精确的。

**提出方法：** Flow-DPPO，用**基于散度的遮罩**替代比率裁剪，直接实现信任域约束。

---

## 2 Preliminaries

### 2.1 流匹配基础

流匹配学习一个连续时间速度场 $v_\theta(x_t, t)$，将样本从简单分布输运到数据分布。给定插值路径：
$$
x_t = \alpha_t x_0 + \sigma_t \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)
$$
模型通过回归目标速度来训练。论文采用线性调度 $\alpha_t = 1-t, \sigma_t = t$（即整流流），此时目标速度退化为 $v = \epsilon - x_0$。

### 2.2 RL 微调流匹配模型的现有范式

**MDP 构建：**
- 状态 $s_k$：$(c, t_k, x_{t_k})$，即提示词、当前时间步、当前潜在表示
- 动作 $a_k$：下一个潜在表示 $x_{t_{k+1}}$
- 策略 $\pi_\theta(a_k | s_k)$：由采样器诱导的条件高斯分布
- 奖励：在终端 $x_0$ 处，由奖励模型给出 $R(x_0, c)$

**SDE 采样器诱导随机策略：**

Flow-GRPO 使用 Flow-SDE 采样器，将确定性 ODE 转化为等边缘分布的 SDE，经欧拉-丸山离散化得到：
$$
x_{t-\Delta t} = x_t + \left[v_\theta + \frac{\sigma_t^2}{2t}(x_t + (1-t)v_\theta)\right]\Delta t + \sigma_t\sqrt{\Delta t}\,\epsilon
$$

CPS 采样器提供另一种选择，更好地保留调度器的插值结构：
$$
x_{t-\Delta t} = (1-(t-\Delta t))\hat{x}_0 + (t-\Delta t)\cos(\tfrac{\eta\pi}{2})\hat{x}_1 + (t-\Delta t)\sin(\tfrac{\eta\pi}{2})\epsilon
$$

两种采样器诱导的每步策略都是高斯分布：
$$
p_\theta(x_{t-\Delta t} | x_t, t, c) = \mathcal{N}\big(x_{t-\Delta t}; \mu_\theta(x_t, t, c), \sigma^2(t) I\big)
$$

**Flow-GRPO 的目标函数：**
$$
\mathcal{L}_{\text{Flow-GRPO}}(\theta) = \mathbb{E}\left[\frac{1}{G}\sum_{i=1}^{G}\frac{1}{K}\sum_{k=1}^{K} \min\left(r^i_k(\theta)\hat{A}^i,\; \text{clip}(r^i_k(\theta), 1-\epsilon, 1+\epsilon)\hat{A}^i\right) \right]
$$
其中优势 $\hat{A}^i$ 通过组内标准化得到，比率 $r^i_k$ 的对数形式为：
$$
\log r^i_k(\theta) = \frac{\|x^i_{t_k-\Delta t} - \mu_{\theta_{old}}\|^2 - \|x^i_{t_k-\Delta t} - \mu_\theta\|^2}{2\sigma^2(t_k)}
$$

---

## 3 Methodology

### 3.1 流匹配的信任域策略优化理论

论文首先为流模型建立了策略改进保证。

**定理 1（性能差异恒等式）：** 对于有限时间步 $K-1$ 的流模型 MDP，任意两个策略 $\pi_\theta$ 和 $\pi_{\theta_{old}}$ 的性能差异分解为：
$$
J(\pi_\theta) - J(\pi_{\theta_{old}}) = \mathcal{L}'_{\theta_{old}}(\pi_\theta) - \Delta(\pi_{\theta_{old}}, \pi_\theta)
$$
其中 $\mathcal{L}'_{\theta_{old}}$ 是一阶代理目标，$\Delta$ 是高阶误差项。

**定理 2（策略改进下界）：** 在流模型 MDP 中，策略改进被下界保证：
$$
J(\pi_\theta) - J(\pi_{\theta_{old}}) \geq \mathcal{L}'_{\theta_{old}}(\pi_\theta) - 2\xi(K-1)(K-2) \cdot D^{\max}_{TV}(\pi_{\theta_{old}} \| \pi_\theta)^2
$$
其中 $\xi$ 是最大绝对奖励。**这个界从理论上证明：约束每步散度就能控制惩罚项，保证单调改进。** 因此，信任域约束 $\max_{\pi_\theta} \mathcal{L}'_{\theta_{old}}(\pi_\theta), \text{s.t. } D^{\max}_{TV} \leq \delta$ 是理论合理的。

**注记 3（高斯设定下的精确散度）：** 在每步高斯策略下，TV 散度是均值位移的单调函数：
$$
D_{TV}(\pi_{\theta_{old}} \| \pi_\theta) = 2\Phi\left(\frac{\|\mu_{\theta_{old}} - \mu_\theta\|}{2\sigma}\right) - 1
$$
因此约束 TV 等价于约束均值距离。同时，Pinsker 不等式 $D^2_{TV} \leq \frac{1}{2}D_{KL}$ 保证了 KL 约束也上界 TV 散度。

### 3.2 比率裁剪在 Flow-GRPO 中的陷阱

这是论文最关键的诊断部分。比率裁剪的条件 $|r^i_t - 1| \leq \epsilon$ 意在防止策略偏离太远，但问题是：

- 按 TV 散度的定义，$D_{TV} = \frac{1}{2}\mathbb{E}_{x_{t-\Delta t}\sim\pi_{\theta_{old}}}[|r-1|]$，每个 $|r^i_t - 1|$ 只是 $2D_{TV}$ 的**单样本估计**。
- 策略改进定理要求约束 $D^{\max}_{TV}$，比率裁剪却约束了这个有噪声的每样本代理量。

**数学分析揭示的病理：** 将采样过程写为 $x^i_{t-\Delta t} = \mu_{\theta_{old}} + \sigma\epsilon$，并令 $d = \mu_\theta - \mu_{\theta_{old}}$，代入对数比率：
$$
\log r^i_t = \frac{\|\sigma\epsilon\|^2 - \|\sigma\epsilon - d\|^2}{2\sigma^2} = \frac{\epsilon^\top d}{\sigma} - \frac{\|d\|^2}{2\sigma^2}
$$

**两个毁灭性结论：**

1.  **高方差：** 第一项 $\epsilon^\top d / \sigma$ 是零均值随机变量，方差为 $\|d\|^2/\sigma^2$。信号项 $-\|d\|^2/(2\sigma^2)$ 恰好是负的 KL 散度，但被同量级的噪声污染。即使真实 KL 散度很小，个别比率样本也可能极端，触发虚假裁剪。

2.  **噪声依赖的裁剪决策：** 更新是否被裁剪，严重取决于采样时的随机噪声 $\epsilon$，而非真实的策略差异。两条有相同策略参数但不同噪声实现的轨迹，可能获得完全不同的裁剪结果。

相比之下，真实 KL 散度 $D_{KL}(\pi_{\theta_{old}}\|\pi_\theta) = \|d\|^2/(2\sigma^2)$ 是策略参数的确定性函数，完全不受采样噪声影响。这构成了用直接散度约束替代比率裁剪的根本动机。

### 3.3 流匹配的散度近端策略优化

**精确 KL 散度的闭式解：** 由于新旧策略都是同方差的高斯分布：
$$
D_{KL}(\pi_{\theta_{old}} \| \pi_\theta) = \frac{\|\mu_{\theta_{old}} - \mu_\theta\|^2}{2\sigma^2}
$$
对于 Flow-SDE 和 CPS 采样器，分别有相应的具体形式（公式 15 和 16）。

**Flow-DPPO 的遮罩设计：** 目标函数为：
$$
\mathcal{L}_{\text{Flow-DPPO}}(\theta) = \mathbb{E}\left[\frac{1}{G}\sum_{i=1}^{G}\frac{1}{T}\sum_{t=0}^{T-1} \left(M^i_t \cdot r^i_t(\theta) \cdot \hat{A}^i - \beta D_{KL}(\pi_\theta \| \pi_{ref})\right) \right]
$$
其中散度遮罩定义为：
$$
M^i_t =
\begin{cases}
0, & \text{若 } (\hat{A}^i > 0 \land r^i_t > 1 \land D_t > \delta) \;\lor\; (\hat{A}^i < 0 \land r^i_t < 1 \land D_t > \delta) \\
1, & \text{否则}
\end{cases}
$$

**不对称设计的关键性质：**

- **当 $\hat{A}^i > 0$ 且 $r^i_t > 1$：** 梯度在推动策略进一步远离旧策略（增大本已增大的动作概率），如果散度已超阈值，遮罩置 0 阻断。
- **当 $\hat{A}^i < 0$ 且 $r^i_t < 1$：** 梯度在进一步远离旧策略（减小本已减小的动作概率），如果散度已超阈值，遮罩置 0 阻断。
- **所有其他情况（$\hat{A}^i > 0, r^i_t < 1$ 或 $\hat{A}^i < 0, r^i_t > 1$）：** 梯度在**向旧策略回归**，这些有益的纠正更新**永不阻断**，不论散度有多大。

这种不对称性保留了 PPO 的有效非对称结构：偏离时受约束，回归时完全自由，使策略在漂移后能快速恢复。

---

## 4 Experiments

### 4.1 主要结果

**实验设置：** 采用 SD3.5 Medium、FLUX2-klein-base-9B、FLUX.1-dev 作为基础模型，覆盖不同架构和规模。对比 Flow-GRPO、Flow-CPS、GRPO-Guard 和 Diffusion-NFT。

**性能和泛化能力：** Flow-DPPO 变体在所有基础模型和评估指标上一致超越所有基线，尤其在 GenEval2 奖励上有显著提升。训练曲线更稳定。

**域外行为与灾难性遗忘：** Flow-DPPO 变体的域外指标（PickScore、CLIP、HPSv2）下降明显更少，表明灾难性遗忘被有效缓解。同时维持更低的 KL 散度偏离，说明散度遮罩充当了安全边界，允许模型从奖励中学习而不失去原始生成质量。

### 4.2 分析

**不对称遮罩与散度阈值的影响：** 
- 去掉不对称掩码后训练崩溃，因为落在信任域外的样本被彻底忽略，优化不再推进。
- 不对称掩码把这些样本约束回信任域，稳定训练。
- 更紧的阈值（$10^{-7}$）初期学习慢，但因更严格的信任域约束，最终收敛稍好于较松的阈值（$10^{-5}$）。

**多轮训练与样本效率：** 基线方法（Flow-GRPO、Flow-CPS）在多轮数据复用下性能趋于饱和甚至退化，而 Flow-DPPO 变体成功跨越多轮更新实现持续改进。这是因为散度遮罩严格限制更新在信任域内，确保安全的样本复用，为采样昂贵的场景（如长视频生成）提供了效率提升的方向。

---

## 5 Conclusion

论文揭示：流模型中的比率裁剪是散度的一个有噪声、有偏的代理。利用流模型高斯策略的结构优势，Flow-DPPO 用零额外成本的精确 KL 散度构建遮罩，直接实现信任域约束。结果是：更优的奖励优化、更好的 KL 近端效率、更强的抗遗忘能力、以及比率裁剪会退化的多轮训练中的稳定性。

## Flow-GRPO与Flow-DPPO损失函数对比

---

### 1. Flow-GRPO 的损失函数（采用比率裁剪）

$$
\boxed{
\mathcal{L}_{\text{Flow-GRPO}}(\theta) = \mathbb{E}\left[ \frac{1}{G}\sum_{i=1}^{G} \frac{1}{K}\sum_{k=1}^{K} \min\left( r^i_k \hat{A}^i,\; \text{clip}(r^i_k, 1-\epsilon, 1+\epsilon)\hat{A}^i \right) - \beta \cdot D_{KL}(\pi_\theta \| \pi_{ref})
\right]
}
$$

**关键点：**
- 这里的 $D_{KL}(\pi_\theta \| \pi_{ref})$ 也是基于**单样本动作 $a_t$** 计算的对数概率比率估计的，和比率裁剪里的 $r^i_k$ 是同一类有噪声的量。
- 它是对**当前策略 $\pi_\theta$** 和**固定的初始预训练模型 $\pi_{ref}$** 之间的 KL 散度进行惩罚，目的是防止模型跑离最初的起点太远（全局软约束）。

**核心机制：** 对每一个时间步的单样本比率 $r^i_t$ 进行**连续数值裁剪**。

---

### 2. Flow-DPPO 的损失函数（采用散度遮罩）

$$
\boxed{
\mathcal{L}_{\text{Flow-DPPO}}(\theta) = \mathbb{E}\left[ \frac{1}{G}\sum_{i=1}^{G} \frac{1}{T}\sum_{t=0}^{T-1} \left( M^i_t \cdot r^i_t(\theta) \cdot \hat{A}^i - \beta \cdot D_{KL}(\pi_\theta \| \pi_{ref}) \right) \right]
}
$$

其中，遮罩 $M^i_t$ 定义为：
$$
M^i_t =
\begin{cases}
0, & \text{若 } \big(\hat{A}^i > 0 \land r^i_t > 1 \land D_t > \delta\big) \;\lor\; \big(\hat{A}^i < 0 \land r^i_t < 1 \land D_t > \delta\big) \\
1, & \text{其他情况}
\end{cases}
$$
而 $D_t = D_{\text{KL}}(\pi_{\theta_{\text{old}}} \| \pi_\theta) = \frac{\|\mu_{\theta_{\text{old}}} - \mu_\theta\|^2}{2\sigma^2}$ 是**精确的、无噪声的KL散度**。

**核心机制：** 用基于**精确分布散度**的二值遮罩 $M^i_t$ 来决定是否将某个样本的更新归零，而不是去连续裁剪比率本身。

**关键点：**
- Flow-DPPO 里的这个 $D_{KL}(\pi_\theta \| \pi_{ref})$ **不再是基于单样本比率的噪声估计**。因为 $\pi_\theta$ 和 $\pi_{ref}$ 都是同方差的高斯分布，所以它同样可以**精确、闭式地计算**为：
  $$
  D_{KL}(\pi_\theta \| \pi_{ref}) = \frac{\|\mu_\theta - \mu_{ref}\|^2}{2\sigma^2}
  $$
  这里 $\mu_{ref}$ 是冻结的参考模型（初始预训练模型）的速度场均值。

---

### 3. 一张表看清差异

| 对比维度 | Flow-GRPO | Flow-DPPO |
| :--- | :--- | :--- |
| **损失函数形式** | 用 $\min(\cdot, \text{clip}(\cdot))$ 包裹单样本比率 | 用二值遮罩 $M^i_t$ 乘以单样本比率 |
| **控制粒度** | **连续**：把比率数值硬压回 $[1-\epsilon, 1+\epsilon]$ | **硬开关**：要么允许更新（1），要么完全阻断（0） |
| **约束的触发信号** | 单样本比率 $r^i_t$ 是否偏离1 | 新旧策略的**整体分布差异** $D_t$ 是否超过阈值 $\delta$ |
| **信号性质** | **有噪声**：$r^i_t$ 严重依赖采样时的随机噪声 | **无噪声**：$D_t$ 是确定性的，由两个分布的均值决定 |
| **是否多条件判定** | 否：仅看 $|r^i_t - 1| > \epsilon$ | **是**：同时参考 $\hat{A}^i$、$r^i_t$ 的方向和 $D_t$ 的值 |
| **对 KL 惩罚的处理** | 通常包含在总损失中（此处省略） | **显式写出**：$-\beta D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}})$ |

---

### 一句话总结

**Flow-GRPO 是对充满噪声的单样本比率做连续裁剪；Flow-DPPO 是用干净的整体分布差异做一个二值开关，来决定样本是否参与更新。**

## Flow-GRPO与Flow-DPPO的KL散度项计算

结论先行：**Flow-GRPO 的 KL 惩罚是一个作用于整条轨迹平均表现的“模糊全局约束”，而 Flow-DPPO 的 KL 惩罚是一个作用于每一步的“精确逐点约束”。** 这背后的原因，正是我们反复讨论的“噪声”问题。

下面我们来详细拆解这两种计算方式。

---

### 1. Flow-GRPO 中的 KL 惩罚计算

在 Flow-GRPO（以及原始的 GRPO）的代码实现中，KL 散度通常表现为**一个标量**，作用于一个生成批次（或组）的所有去噪步骤。

#### 它是怎么计算的？
它是通过**采样估计**的，依赖的是那个充满噪声的单样本动作 $a_t$。
1.  **记录概率**：在采样阶段，记录旧策略 $\pi_{old}$ 在每个时间步 $t$ 对采样动作 $a_t$ 的对数概率 $\log \pi_{old}(a_t|s_t)$。
2.  **计算新概率**：在训练阶段，用新策略 $\pi_\theta$ 计算同一动作 $a_t$ 的对数概率 $\log \pi_\theta(a_t|s_t)$。
3.  **逐点估计**：每个采样动作的对数概率差 $\log \pi_{old} - \log \pi_\theta$ 就是对**该点** KL 散度的一个单样本、无偏但高方差的估计。
4.  **全局平均**：然后，它会把**一个组内所有样本在所有时间步上的这些估计值进行平均**，得到一个单一的标量作为 KL 惩罚。

这个过程可以表示为：
$$
\text{KL}_{\text{GRPO}} \approx \frac{1}{G \times K} \sum_{i=1}^{G} \sum_{t=1}^{K} \left[ \log \pi_{old}(a_t^i|s_t^i) - \log \pi_{\theta}(a_t^i|s_t^i) \right]
$$

#### 为什么写成放到期望外面？
正因为它是这种“全局平均”的标量，在损失函数里，它实际上可以放在所有求和和期望符号的外面，作为一个独立的、整体性的惩罚项。论文里虽然省略了，但意思就是：**整个批次的损失 = 策略损失 - $\beta$ * 这个平均KL估计值**。

---

### 2. Flow-DPPO 中的 KL 惩罚计算

Flow-DPPO 的做法则完全不同，它利用的是高斯策略的结构优势。

#### 它是怎么计算的？
它**直接计算**，完全不依赖采样的动作 $a_t$。
1.  **获取均值**：对于任意状态 $s_t$，新旧策略分别是高斯分布 $\mathcal{N}(\mu_{\theta_{old}}, \sigma^2 I)$ 和 $\mathcal{N}(\mu_{\theta}, \sigma^2 I)$。
2.  **精确计算**：直接用两个分布的均值计算闭式解：
    $$
    D_{KL}(\pi_{\theta_{old}} \| \pi_{\theta}) = \frac{\|\mu_{\theta_{old}}(s_t) - \mu_\theta(s_t)\|^2}{2\sigma^2(t)}
    $$
3.  **同样用于参考模型**：对于指向 $\pi_{ref}$ 的惩罚项，计算方式完全一样：
    $$
    D_{KL}(\pi_{\theta} \| \pi_{ref}) = \frac{\|\mu_{\theta}(s_t) - \mu_{ref}(s_t)\|^2}{2\sigma^2(t)}
    $$

#### 为什么它被放在最内层循环里？
因为这个计算**不需要对轨迹或组进行平均来消除噪声**，它本身就是确定性、无噪声的。这意味着它能以极高的分辨率工作：

-   **它就是每一步的散度**：这个公式算出的就是“在时间步 $t$，对于第 $i$ 个样本，策略偏离参考模型”的精确值。
-   **能实现精细控制**：既然有了每一步的精确值，最优的做法就是把它放在最内层，对每一步的更新都进行精确地牵引。如果只在最后算一个全局平均，就浪费了这种精确性带来的控制力。

所以，在 Flow-DPPO 的损失函数里，它被显式地写在内部的求和项中，表示**对每一步都施加一个精确、独立的 KL 惩罚**。

---

### 两者的本质区别对比

| 对比维度 | Flow-GRPO 的 KL 惩罚 | Flow-DPPO 的 KL 惩罚 |
| :--- | :--- | :--- |
| **计算方式** | **基于采样的估计**：$ \log \pi_{old}(a_t) - \log \pi_{\theta}(a_t) $ | **基于分布的解析计算**：$ \frac{\|\mu_{\theta_{old}} - \mu_\theta\|^2}{2\sigma^2} $ |
| **数据依赖** | 依赖于采样时的随机动作 $a_t$ | 不依赖动作 $a_t$，仅依赖网络输出 |
| **信号质量** | **有噪声、高方差**的单样本估计 | **无噪声、确定性**的精确值 |
| **作用范围** | 需要通过组内和时间步的**平均**来降噪，是一个**全局标量** | 无需平均即可独立存在，是一个**逐点张量** |
| **约束精度** | **模糊**：只知道整个轨迹平均偏离了多少 | **精确**：知道轨迹上每一步的偏离情况，实现逐点控制 |
| **在损失中的位置**| 求和期望之外，作为全局惩罚项 | 最内层求和之内，作为逐点惩罚项 |

**一句话总结：GRPO 因为用充满噪声的样本来估计，所以被迫把 KL 惩罚设计成一个粗糙的“全局平均约束”。而 Flow-DPPO 拥有精确计算的能力，因此它把 KL 惩罚精确地施加在**每一个决策点**上，做到了精细的“逐点约束”。**